In [7]:
!pip install -q ultralytics mediapipe opencv-python

In [6]:
import cv2
import mediapipe as mp  # 스켈레톤을 추출하는 라이브러리
import numpy as np
import csv  # csv 저장을 위해 라이브러리 추가
import os   # 파일 경로 관리를 위해 라이브러리 추가

# MediaPipe Pose 모델 초기화
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(
    min_detection_confidence = 0.7,  # 감지 최소 신뢰도
    min_tracking_confidence = 0.3 # 추적 최소 신뢰도
)

# MediaPipe 그리기 유틸리티 초기화
mp_drawing = mp.solutions.drawing_utils

# 동영상 파일 경로
video_path = "aespa_test.mp4" # 분석할 동영상 파일 경로 입력
cap = cv2.VideoCapture(video_path)

# 출력 파일 이름 설정
output_filename = os.path.splitext(os.path.basename(video_path))[0] + "_skeleton.csv"
# CSV 파일 헤더 준비 (33개 랜드마크 * 4개 좌표)
landmarks = ['class'] + [f'{j}_{i}' for i in mp_pose.PoseLandmark._member_names_ for j in ('x', 'y', 'z', 'v')]

# 동영상 파일이 정상적으로 열렸는지 확인
if not cap.isOpened():
    print(f"오류: '{video_path}' 동영상을 열 수 없습니다.")
    exit()

print("스켈레톤 추출을 시작합니다. 종료하려면 'q' 키를 누르세요.")

with open(output_filename, 'w', newline='') as f:
    csv_writer = csv.writer(f)
    csv_writer.writerow(landmarks)  # 헤더 작성

    print(f"'{output_filename}' 파일에 스켈레톤 데이터 저장을 시작합니다.")
    frame_count = 0
    while cap.isOpened():
        # 동영상에서 프레임 읽기
        success, image = cap.read()

        if not success:
            print("동영상 스트림의 끝에 도달했거나 오류가 발생했습니다.")
            break

        # 성능 향상을 위해 이미지를 읽기 전용으로 표시
        image.flags.writeable = False
        # BGR 이미지를 RGB로 변환
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # MediaPipe Pose를 사용하여 포즈 감지 수행
        results = pose.process(image_rgb)

        # 이미지를 다시 쓰기 가능으로 변경
        image.flags.writeable = True

        # 감지된 스켈레톤(포즈 랜드마크)을 원본 이미지에 그리기
        if results.pose_landmarks:
            mp_drawing.draw_landmarks(
                image,
                results.pose_landmarks,
                mp_pose.POSE_CONNECTIONS,
                landmark_drawing_spec = mp_drawing.DrawingSpec(color = (245, 117, 66), thickness=2, circle_radius=2),
                connection_drawing_spec=mp_drawing.DrawingSpec(color=(245, 66, 230), thickness=2, circle_radius=2)
            )

            # 랜드마크 데이터 추출 및 csv 행으로 변환
            try:
                # 'dance' 클래스로 분류 (필요에 따라 변경 가능)
                class_name = "dance"

                # 모든 랜드마크의 x, y, z, v 값을 순서대로 리스트에 담기
                pose_row = list(np.array([[res.x, res.y, res.z, res.visibility] for res in results.pose_landmarks.landmark]).flatten())

                # 클래스 이름과 랜드마크 데이터를 합쳐서 한 행으로 만듦
                row = [class_name] + pose_row
                
                # csv 파일에 한 행 쓰기
                csv_writer.writerow(row)

            except Exception as e:
                print(f"프레임 {frame_count} 처리 중 오류 발생: {e}")
                pass # 오류 발생 시 해당 프레임 건너뜀

        # 결과 영상 출력
        cv2.imshow('MediaPipe Pose Skeleton', image)

        frame_count += 1
        # 'q' 키를 누르면 루프 종료
        if cv2.waitKey(5) & 0xFF == ord('q'):
            break

# 자원 해제
cap.release()
cv2.destroyAllWindows()
pose.close()

print("스켈레톤 추출이 완료되었습니다.")

스켈레톤 추출을 시작합니다. 종료하려면 'q' 키를 누르세요.
'aespa_test_skeleton.csv' 파일에 스켈레톤 데이터 저장을 시작합니다.
스켈레톤 추출이 완료되었습니다.


In [8]:
! pip install scipy

In [9]:
# mediapipe 라이브러리를 사용하여 기본 스켈레톤 추출
# YOLO 라이브러리를 사용하여, 다중 객체 탐지 (Top_down 방식)
# 칼만 필터(Kalman Filter)를 사용하여, 각 관절의 다음 위치를 예측하고 보정하여 성능 개선
# 상태 예측 및 보간: 추적을 잠시 놓친 스켈레톤의 위치를 예측하여, 시각적 끊김 없이 보이도록 처리
# 슬롯 기반 할당(Slot-Based Assignment)' 개념을 도입하여 성능 향상
# 헝가리안 알고리즘(scipy.linear_sum_assignment)을 사용:
# - 매 프레임마다 새로 탐지된 사람들과 기존 '댄서 슬롯'들의 위치를 비교하여, 전체적으로 가장 거리가 가까운 최적의 짝을 확인
# 3D 칼만 필터: 개별 댄서의 물리적 움직임을 3차원 공간에서 부드럽고 정확하게 예측

import cv2
import mediapipe as mp
from ultralytics import YOLO
import numpy as np
import random
import csv
from collections import Counter
from scipy.optimize import linear_sum_assignment

# ----------------------------------
# 1. 클래스 및 함수 정의
# ----------------------------------
class KalmanFilter3D:
    """A simple Kalman filter for 3D point tracking."""
    def __init__(self, dt=1, std_acc=1, x_std_meas=0.1, y_std_meas=0.1, z_std_meas=0.1):
        self.state = np.zeros((6, 1))   # 상태 변수를 6차원으로 확장: [x, y, z, vx, vy, vz]
        # 상태 전이 행렬을 6x6으로 확장
        self.F = np.array([[1,0,0,dt,0,0], [0,1,0,0,dt,0], [0,0,1,0,0,dt],
                           [0,0,0,1,0,0], [0,0,0,0,1,0], [0,0,0,0,0,1]])
        # 측정 행렬을 3x6으로 확장
        self.H = np.array([[1,0,0,0,0,0], [0,1,0,0,0,0], [0,0,1,0,0,0]])
        self.Q = np.eye(6)*std_acc**2
        self.R = np.diag([x_std_meas**2, y_std_meas**2, z_std_meas**2])
        self.P = np.eye(6)
    def predict(self):
        self.state = np.dot(self.F, self.state)
        self.P = np.dot(np.dot(self.F, self.P), self.F.T) + self.Q
        return self.state
    def update(self, z):
        # 입력 z는 이제 3D 벡터 [x, y, z]
        S = np.dot(self.H, np.dot(self.P, self.H.T)) + self.R
        K = np.dot(np.dot(self.P, self.H.T), np.linalg.inv(S))
        self.state += np.dot(K, (z - np.dot(self.H, self.state)))
        self.P -= np.dot(np.dot(K, self.H), self.P)
        return self.state

class DancerSlot:
    """영구적인 댄서 슬롯의 모든 데이터를 저장하는 클래스"""
    def __init__(self, slot_id, initial_bbox):
        self.id = slot_id
        self.kalman_filters = [KalmanFilter3D() for _ in range(33)]
        self.landmarks = np.zeros((33, 4))
        self.bbox = initial_bbox
        self.disappeared_frames = 0
        self.is_active = True
        self.color = (random.randint(0, 255), random.randint(0, 255), random.randint(0, 255))

    def reinitialize_kalman_filters(self):
        """칼만 필터를 초기 상태로 리셋합니다."""
        self.kalman_filters = [KalmanFilter3D() for _ in range(33)]

# ----------------------------------
# 2. 초기 설정
# ----------------------------------
TARGET_PERSON_COUNT = 1
yolo_model = YOLO('yolov8m.pt')
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=True, min_detection_confidence=0.5)
video_path = "TWICE_ICANTSTOPME.mp4"
cap = cv2.VideoCapture(video_path)

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)); fps = int(cap.get(cv2.CAP_PROP_FPS))
output_video_path = video_path.replace('.mp4', '_output.mp4')
out = cv2.VideoWriter(output_video_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (frame_width, frame_height))
csv_output_path = video_path.replace('.mp4', '_skeletons_data.csv')
csv_file = open(csv_output_path, 'w', newline='', encoding='utf-8')
csv_writer = csv.writer(csv_file)
csv_writer.writerow(['frame', 'slot_id', 'landmark_id', 'x', 'y', 'z', 'visibility', 'bbox_x1', 'bbox_y1', 'bbox_x2', 'bbox_y2'])

dancer_slots = {}
MAX_DISAPPEARED_FRAMES = 15
frame_counter = 0

print(f"3D 칼만 필터 기반 추적 시스템을 시작합니다. 목표 인원수: {TARGET_PERSON_COUNT}명")

# ----------------------------------
# 3. 메인 루프: 프레임별 처리
# ----------------------------------
while cap.isOpened():
    success, frame = cap.read()
    if not success: break

    # STEP 1: YOLO로 모든 사람 '탐지' (track 대신 predict 사용)
    results = yolo_model.predict(frame, classes=[0], conf=0.4, verbose=False)
    detections = results[0].boxes.xyxy.cpu().numpy()

    # --- 슬롯 초기화 로직 ---
    if len(dancer_slots) < TARGET_PERSON_COUNT:
        for i, bbox in enumerate(detections):
            # 이미 존재하는 슬롯의 개수를 ID로 사용
            slot_id = len(dancer_slots)
            if slot_id < TARGET_PERSON_COUNT:
                print(f"프레임 {frame_counter}: 댄서 슬롯 {slot_id} 초기화.")
                dancer_slots[slot_id] = DancerSlot(slot_id, bbox)
            else:
                break
    
    # --- 슬롯 할당 로직 (매 프레임 실행) ---
    matched_detection_indices = set()
    if detections.shape[0] > 0 and len(dancer_slots) > 0:
        # 1. 활성화된 슬롯과 모든 탐지 객체 간의 매칭
        active_slots = {sid: slot for sid, slot in dancer_slots.items() if slot.is_active}
        slot_keys = list(active_slots.keys())
        
        if len(slot_keys) > 0:
            # 비용 행렬 계산
            cost_matrix = np.zeros((len(slot_keys), len(detections)))
            for i, slot_id in enumerate(slot_keys):
                slot_center = ((active_slots[slot_id].bbox[0] + active_slots[slot_id].bbox[2]) / 2, (active_slots[slot_id].bbox[1] + active_slots[slot_id].bbox[3]) / 2)
                for j, det_box in enumerate(detections):
                    det_center = ((det_box[0] + det_box[2]) / 2, (det_box[1] + det_box[3]) / 2)
                    cost_matrix[i, j] = np.linalg.norm(np.array(slot_center) - np.array(det_center))
            
            # 헝가리안 알고리즘으로 최적 할당
            row_ind, col_ind = linear_sum_assignment(cost_matrix)
            
            # 할당된 슬롯 업데이트
            for r, c in zip(row_ind, col_ind):
                if cost_matrix[r, c] < 250: # 너무 멀리 떨어진 매칭은 무시
                    slot_id = slot_keys[r]
                    person_slot = dancer_slots[slot_id]
                    person_slot.disappeared_frames = 0
                    person_slot.is_active = True
                    person_slot.bbox = detections[c]
                    matched_detection_indices.add(c)

    # 2. ID 부활 로직: 비활성화된 슬롯과 매칭되지 않은 탐지 객체 간의 매칭
    unmatched_detection_indices = list(set(range(len(detections))) - matched_detection_indices)
    inactive_slots = {sid: slot for sid, slot in dancer_slots.items() if not slot.is_active}

    if len(unmatched_detection_indices) > 0 and len(inactive_slots) > 0:
            unmatched_detections = detections[unmatched_detection_indices]
            inactive_slot_keys = list(inactive_slots.keys())

            # 비활성 슬롯과 남은 탐지 객체 간의 실제 거리로 비용 행렬 계산
            cost_matrix_revival = np.zeros((len(inactive_slot_keys), len(unmatched_detections)))
            for i, slot_id in enumerate(inactive_slot_keys):
                # 비활성 슬롯의 마지막 예측 위치를 사용
                slot_center = ((inactive_slots[slot_id].bbox[0] + inactive_slots[slot_id].bbox[2]) / 2, (inactive_slots[slot_id].bbox[1] + inactive_slots[slot_id].bbox[3]) / 2)
                for j, det_box in enumerate(unmatched_detections):
                    det_center = ((det_box[0] + det_box[2]) / 2, (det_box[1] + det_box[3]) / 2)
                    cost_matrix_revival[i, j] = np.linalg.norm(np.array(slot_center) - np.array(det_center))

            row_ind_rev, col_ind_rev = linear_sum_assignment(cost_matrix_revival)
            for r, c in zip(row_ind_rev, col_ind_rev):
                        slot_id = inactive_slot_keys[r]
                        detection_idx = list(unmatched_detection_indices)[c]
                        
                        print(f"프레임 {frame_counter}: 슬롯 {slot_id} 부활!")
                        person_slot = dancer_slots[slot_id]
                        person_slot.is_active = True
                        person_slot.disappeared_frames = 0
                        person_slot.bbox = detections[detection_idx]
                        person_slot.reinitialize_kalman_filters() # 칼만 필터 초기화
                        matched_detection_indices.add(detection_idx)

    # --- 3. 포즈 추정 및 상태 업데이트/예측 ---
    for slot_id, person_slot in dancer_slots.items():
        if not person_slot.is_active: continue

        # 현재 프레임에서 업데이트된(매칭된) 슬롯인 경우
        if person_slot.disappeared_frames == 0:
            x1, y1, x2, y2 = [int(coord) for coord in person_slot.bbox]
            padding = 0.15; box_w, box_h = x2 - x1, y2 - y1
            x1_pad = max(0, int(x1 - box_w * padding)); y1_pad = max(0, int(y1 - box_h * padding))
            x2_pad = min(frame_width, int(x2 + box_w * padding)); y2_pad = min(frame_height, int(y2 + box_h * padding))
            person_crop = frame[y1_pad:y2_pad, x1_pad:x2_pad]

            if person_crop.shape[0] > 0 and person_crop.shape[1] > 0:
                crop_rgb = cv2.cvtColor(person_crop, cv2.COLOR_BGR2RGB); pose_results = pose.process(crop_rgb)
                if pose_results.pose_landmarks and len(pose_results.pose_landmarks.landmark) == 33:
                    crop_h, crop_w, _ = person_crop.shape
                    for i, landmark in enumerate(pose_results.pose_landmarks.landmark):
                        measured_x, measured_y = x1_pad + landmark.x * crop_w, y1_pad + landmark.y * crop_h
                        measured_z = landmark.z * crop_w 
                        kf = person_slot.kalman_filters[i]
                        if np.all(kf.state[0:3] == 0): kf.state[0], kf.state[1], kf.state[2] = measured_x, measured_y, measured_z
                        kf.predict(); updated_state = kf.update(np.array([[measured_x], [measured_y], [measured_z]]))
                        person_slot.landmarks[i] = [updated_state[0, 0], updated_state[1, 0], updated_state[2, 0], landmark.visibility]
        else: # 사라진 슬롯인 경우
            person_slot.disappeared_frames += 1
            if person_slot.disappeared_frames > MAX_DISAPPEARED_FRAMES:
                print(f"프레임 {frame_counter}: 슬롯 {slot_id} 비활성화.")
                person_slot.is_active = False
                continue
            
            for i in range(33):
                kf = person_slot.kalman_filters[i]; predicted_state = kf.predict()
                person_slot.landmarks[i, 0:3] = predicted_state[0:3, 0].T
            
            center_x = np.mean(person_slot.landmarks[:, 0]); center_y = np.mean(person_slot.landmarks[:, 1])
            if not (np.isnan(center_x) or np.isnan(center_y)):
                w, h = person_slot.bbox[2] - person_slot.bbox[0], person_slot.bbox[3] - person_slot.bbox[1]
                person_slot.bbox = [center_x - w/2, center_y - h/2, center_x + w/2, center_y + h/2]

    # --- 4. 최종 시각화 및 데이터 저장 ---
    for slot_id, person_slot in dancer_slots.items():
        if not person_slot.is_active: continue

        color = person_slot.color; label = f"ID: {person_slot.id}"
        if person_slot.disappeared_frames > 0:
            color = tuple(c // 2 for c in color)
            label += " (Lost)"
        x1, y1, x2, y2 = [int(c) for c in person_slot.bbox]
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
        
        landmarks_to_process = person_slot.landmarks
        bbox_for_csv = person_slot.bbox.tolist()    # 후처리를 위해 bbox 좌표도 함께 저장

        for i, landmark in enumerate(landmarks_to_process):
            csv_writer.writerow([frame_counter, person_slot.id, i, landmark[0], landmark[1], landmark[2], landmark[3]] + bbox_for_csv)
            if landmark[3] > 0.5:
                cv2.circle(frame, (int(landmark[0]), int(landmark[1])), 3, color, -1)

        connections = mp_pose.POSE_CONNECTIONS
        for connection in connections:
            start_idx, end_idx = connection
            if landmarks_to_process[start_idx, 3] > 0.5 and landmarks_to_process[end_idx, 3] > 0.5:
                start_point = (int(landmarks_to_process[start_idx, 0]), int(landmarks_to_process[start_idx, 1]))
                end_point = (int(landmarks_to_process[end_idx, 0]), int(landmarks_to_process[end_idx, 1]))
                cv2.line(frame, start_point, end_point, color, 2)

    out.write(frame)
    cv2.imshow('Slot-Based High-Performance Tracking', frame)
    frame_counter += 1
    if cv2.waitKey(1) & 0xFF == ord('q'): break

# ----------------------------------
# 4. 종료 처리
# ----------------------------------
csv_file.close(); cap.release(); out.release(); cv2.destroyAllWindows(); pose.close()
print(f"처리 완료! 최종 결과가 '{output_video_path}'와 '{csv_output_path}'에 저장되었습니다.")


3D 칼만 필터 기반 추적 시스템을 시작합니다. 목표 인원수: 1명
프레임 29: 댄서 슬롯 0 초기화.
처리 완료! 최종 결과가 'TWICE_ICANTSTOPME_output.mp4'와 'TWICE_ICANTSTOPME_skeletons_data.csv'에 저장되었습니다.


In [17]:
!pip install networkx

In [20]:
# 후처리1
# 트랙릿 재연결 (ID 스위치 보정): slot_id의 이동 경로를 하나의 '트랙릿(Tracklet)'으로 간주
#    → 각 트랙릿의 특정 지점(예: 경로가 끊기기 직전, ID가 바뀐 직후)에서 **외모 특징(옷 색상)**과 **움직임 특징(위치, 속도)**을 추출
#    → 한 트랙릿이 끝나는 지점과 다른 트랙릿이 시작하는 지점의 특징들을 비교
#    → 만약 "A 트랙릿의 끝과 B 트랙릿의 시작이 동일 인물일 확률이 매우 높다"고 판단되면, 두 트랙릿을 하나로 합쳐 ID를 통일

import pandas as pd
import numpy as np
import cv2
from scipy.optimize import linear_sum_assignment
from collections import defaultdict
import time
import networkx as nx
from collections import Counter

# -- 설정 --
# 생성된 원본 데이터 파일
INPUT_CSV_PATH = 'TWICE_ICANTSTOPME_skeletons_data.csv'

# 원본 비디오 파일 (외모 특징 추출에 필요)
VIDEO_PATH = 'TWICE_ICANTSTOPME.mp4'

# 후처리가 완료된 데이터가 저장될 파일
OUTPUT_CSV_PATH = 'TWICE_ICANTSTOPME_skeletons_postprocessed.csv'

# -- 파라미터 --
JUMP_THRESHOLD = 150  # ID 스위치를 의심할 위치 '점프' 픽셀 거리, 값이 작을 수록 더 꼼꼼하게 검사
MAX_LINK_FRAMES = 45  # 두 트랙릿을 연결할 최대 프레임 간격
APPEARANCE_WEIGHT = 0.7 # 외모 특징의 가중치 (0~1), 의상이 비슷하면 값을 낮추기
MOTION_WEIGHT = 0.3     # 움직임 특징의 가중치 (0~1)
COST_THRESHOLD = 0.65   # 연결을 위한 비용 임계값

DISTANCE_GATE_THRESHOLD = 100   # 물리적 거리 제한 추가


# --- 2. 헬퍼 함수 ---
def get_color_histogram(frame, bbox):
    """지정된 바운딩 박스 영역에서 컬러 히스토그램을 계산합니다."""
    x1, y1, x2, y2 = [int(c) for c in bbox]
    roi = frame[y1:y2, x1:x2]
    if roi.size == 0: return None
    hsv_roi = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist([hsv_roi], [0], None, [180], [0, 180])
    cv2.normalize(hist, hist, 0, 1, cv2.NORM_MINMAX)
    return hist.flatten()

# --- 3. 메인 후처리 로직 ---
def relink_switched_ids_final_corrected(csv_path, video_path, output_path):
    start_time = time.time()
    print("후처리 1단계 시작: 자동화된 ID 스위치 보정 (오류 수정 최종본)")
    df = pd.read_csv(csv_path)
    cap = cv2.VideoCapture(video_path)

    # 💡 [오류 수정] 1. 올바른 트랙릿 분리 로직
    print("  - 1/5: 트랙릿 분리 로직 수정 및 실행 중...")
    df['tracklet_id'] = -1
    tracklet_counter = 0
    
    # 원본 slot_id별로 순회
    for slot_id in df['slot_id'].unique():
        person_df = df[df['slot_id'] == slot_id].sort_values(by='frame')
        if person_df.empty: continue

        # 코(landmark_id=0)의 위치를 기준으로 점프가 일어나는 '프레임 번호'를 찾음
        nose_df = person_df[person_df['landmark_id'] == 0]
        if len(nose_df) < 2:
            # 트랙릿이 너무 짧으면 하나의 그룹으로 간주
            df.loc[person_df.index, 'tracklet_id'] = tracklet_counter
            tracklet_counter += 1
            continue
            
        positions = nose_df[['x', 'y']].values
        distances = np.linalg.norm(positions[1:] - positions[:-1], axis=1)
        # 점프가 일어난 행의 index가 아니라, '프레임 번호'를 저장
        jump_frames = nose_df.iloc[np.where(distances > JUMP_THRESHOLD)[0] + 1]['frame'].tolist()
        
        # 트랙릿의 시작과 끝 프레임 번호를 정의
        tracklet_start_frame = person_df['frame'].min()
        for jump_frame in jump_frames:
            # 이전 트랙릿에 ID 할당
            df.loc[(df['slot_id'] == slot_id) & (df['frame'] >= tracklet_start_frame) & (df['frame'] < jump_frame), 'tracklet_id'] = tracklet_counter
            tracklet_counter += 1
            tracklet_start_frame = jump_frame
        
        # 마지막 트랙릿에 ID 할당
        df.loc[(df['slot_id'] == slot_id) & (df['frame'] >= tracklet_start_frame), 'tracklet_id'] = tracklet_counter
        tracklet_counter += 1
        
    # 2. 각 트랙릿의 시작/끝 정보 수집 (이전과 동일)
    print("  - 2/5: 각 트랙릿의 시작/끝 정보 수집 중...")
    tracklet_info = defaultdict(dict)
    for tid, group in df.groupby('tracklet_id'):
        start_frame_info = group.sort_values(by='frame').iloc[0]
        end_frame_info = group.sort_values(by='frame').iloc[-1]
        tracklet_info[tid] = {
            'original_slot_id': start_frame_info['slot_id'], 'start_frame': start_frame_info['frame'], 'end_frame': end_frame_info['frame'],
            'start_motion': start_frame_info[['x', 'y']].values, 'end_motion': end_frame_info[['x', 'y']].values,
            'start_bbox': start_frame_info[['bbox_x1', 'bbox_y1', 'bbox_x2', 'bbox_y2']].values, 'end_bbox': end_frame_info[['bbox_x1', 'bbox_y1', 'bbox_x2', 'bbox_y2']].values,
            'start_appearance': None, 'end_appearance': None
        }

    # 3. 'Single-Pass'로 외모 특징 추출 (이전과 동일)
    print("  - 3/5: 비디오 단일 스캔으로 모든 외모 특징 추출 중...")
    features_to_extract = defaultdict(list)
    for tid, info in tracklet_info.items():
        if info['start_frame'] == info['end_frame']: # 단일 프레임 트랙릿 처리
             features_to_extract[info['start_frame']].append({'tid': tid, 'role': 'single', 'bbox': info['start_bbox']})
        else:
            features_to_extract[info['start_frame']].append({'tid': tid, 'role': 'start', 'bbox': info['start_bbox']})
            features_to_extract[info['end_frame']].append({'tid': tid, 'role': 'end', 'bbox': info['end_bbox']})
    frame_idx = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        if frame_idx in features_to_extract:
            for item in features_to_extract[frame_idx]:
                hist = get_color_histogram(frame, item['bbox'])
                if hist is not None:
                    if item['role'] == 'single':
                        tracklet_info[item['tid']]['start_appearance'] = hist
                        tracklet_info[item['tid']]['end_appearance'] = hist
                    else:
                        tracklet_info[item['tid']][f"{item['role']}_appearance"] = hist
        frame_idx += 1
    cap.release()
    
    # 4. 비용 행렬 계산 및 연결 그래프 생성 (이전과 동일)
    print("  - 4/5: 트랙릿 간 연결 그래프 생성 중...")
    tids = list(tracklet_info.keys())
    cost_matrix = np.full((len(tids), len(tids)), np.inf)
    for i, tid1 in enumerate(tids):
        for j, tid2 in enumerate(tids):
            if i == j: continue
            end_info = tracklet_info[tid1]; start_info = tracklet_info[tid2]
            frame_gap = start_info['start_frame'] - end_info['end_frame']
            if 0 < frame_gap < MAX_LINK_FRAMES:
                if end_info['end_appearance'] is not None and start_info['start_appearance'] is not None:
                    motion_cost = np.linalg.norm(end_info['end_motion'] - start_info['start_motion'])
                    if motion_cost > DISTANCE_GATE_THRESHOLD: continue
                    appearance_cost = 1 - cv2.compareHist(end_info['end_appearance'], start_info['start_appearance'], cv2.HISTCMP_CORREL)
                    norm_motion_cost = min(motion_cost / DISTANCE_GATE_THRESHOLD, 1.0)
                    total_cost = (MOTION_WEIGHT * norm_motion_cost) + (APPEARANCE_WEIGHT * appearance_cost)
                    cost_matrix[i, j] = total_cost
    
    cost_matrix[np.isinf(cost_matrix)] = 1e6
    row_ind, col_ind = linear_sum_assignment(cost_matrix)

    # 5. ID 재할당 ('가장 긴 트랙릿' 기반)
    print("  - 5/5: 연결 그룹 분석 및 최종 ID 할당 중...")
    G = nx.Graph()
    for r, c in zip(row_ind, col_ind):
        if cost_matrix[r, c] < COST_THRESHOLD:
            G.add_edge(tids[r], tids[c])
            
    connected_components = list(nx.connected_components(G))
    final_id_map = {}
    
    for component in connected_components:
        longest_tracklet_id = -1; max_length = -1
        for tid in component:
            length = tracklet_info[tid]['end_frame'] - tracklet_info[tid]['start_frame']
            if length > max_length:
                max_length = length; longest_tracklet_id = tid
        if longest_tracklet_id != -1:
            final_id = tracklet_info[longest_tracklet_id]['original_slot_id']
            for tid in component: final_id_map[tid] = final_id

    for tid in tids:
        if tid not in final_id_map:
            final_id_map[tid] = tracklet_info[tid]['original_slot_id']
            
    df['slot_id'] = df['tracklet_id'].map(final_id_map)
    df.dropna(subset=['slot_id'], inplace=True)
    df['slot_id'] = df['slot_id'].astype(int)
    df.drop(columns=['tracklet_id'], inplace=True)
    
    df.to_csv(output_path, index=False)
    
    print(f"ID가 보정된 데이터가 '{output_path}'에 저장되었습니다.")

# --- 스크립트 실행 ---
if __name__ == "__main__":
    relink_switched_ids_final_corrected(INPUT_CSV_PATH, VIDEO_PATH, OUTPUT_CSV_PATH)

후처리 1단계 시작: 자동화된 ID 스위치 보정 (오류 수정 최종본)
  - 1/5: 트랙릿 분리 로직 수정 및 실행 중...
  - 2/5: 각 트랙릿의 시작/끝 정보 수집 중...
  - 3/5: 비디오 단일 스캔으로 모든 외모 특징 추출 중...
  - 4/5: 트랙릿 간 연결 그래프 생성 중...
  - 5/5: 연결 그룹 분석 및 최종 ID 할당 중...
ID가 보정된 데이터가 'TWICE_ICANTSTOPME_skeletons_postprocessed.csv'에 저장되었습니다.


In [21]:
# 후처리 2
# 인체 모델 제약 적용
# 평균 뼈 길이 계산: 먼저, ID가 보정된 데이터 전체를 분석하여 각 사람(slot_id)의 각 뼈(예: '어깨-팔꿈치', '무릎-발목')의 평균 3D 길이를 계산하여
# 이 평균값은 해당 인물의 '표준 뼈 길이'로 정의
# 오류 탐지 및 보정: 데이터의 모든 프레임을 다시 검토하면서, 특정 프레임의 뼈 길이가 이 '표준 뼈 길이'에서 크게 벗어나는 경우를 '오류'로 간주
# 기구학적 수정 (Kinematic Correction): 오류가 발견되면, 더 안정적인 관절(예: 팔꿈치)을 기준으로, 덜 안정적인 관절(예: 손목)의 위치를 표준 뼈 길이에 맞게 강제로 조정

import pandas as pd
import numpy as np
import mediapipe as mp
from collections import defaultdict

# --- 1. 설정 및 파라미터 ---
# 이전 단계(ID 재연결)에서 생성된 데이터 파일
INPUT_CSV_PATH = 'TWICE_ICANTSTOPME_skeletons_postprocessed.csv'
# 최종적으로 보정된 데이터가 저장될 파일
OUTPUT_CSV_PATH = 'TWICE_ICANTSTOPME_skeletons_data_final_corrected.csv'

# 파라미터
BONE_LENGTH_THRESHOLD_RATIO = 0.20  # 평균 뼈 길이에서 25% 이상 벗어나면 오류로 간주
VISIBILITY_THRESHOLD = 0.6          # 이 값보다 낮으면 가려진 것으로 간주하고 대칭성 적용

# --- 2. 헬퍼 함수 및 데이터 정의 ---
# MediaPipe의 관절 연결 정보
POSE_CONNECTIONS = mp.solutions.pose.POSE_CONNECTIONS
# 뼈 이름과 연결된 관절 ID 정의 (더 명확한 관리를 위해)
BONE_MAPPING = {
    'LEFT_ARM': (11, 13), 'RIGHT_ARM': (12, 14),
    'LEFT_FOREARM': (13, 15), 'RIGHT_FOREARM': (14, 16),
    'LEFT_UPPER_LEG': (23, 25), 'RIGHT_UPPER_LEG': (24, 26),
    'LEFT_LOWER_LEG': (25, 27), 'RIGHT_LOWER_LEG': (26, 28)
}
# 대칭이 되는 관절 쌍 정의 (왼쪽, 오른쪽)
SYMMETRIC_PAIRS = [
    (11, 12), (13, 14), (15, 16), (17, 18), (19, 20), (21, 22), 
    (23, 24), (25, 26), (27, 28), (29, 30), (31, 32)
]

# --- 3. 헬퍼 함수 ---
def get_3d_distance(p1, p2):
    return np.linalg.norm(p1 - p2) if p1 is not None and p2 is not None else 0

# --- 4. 메인 후처리 로직 ---
def apply_human_model_constraints(csv_path, output_path):
    print("후처리 2단계 시작: 인체 모델 제약 적용 (최종 완전판)")
    df = pd.read_csv(csv_path)
    df_pivot = df.pivot_table(index=['frame', 'slot_id'], columns='landmark_id', values=['x', 'y', 'z', 'visibility'])

    # 1. 각 사람의 평균 '신체 비율' 계산
    print("  - 1/3: 각 인물의 평균 신체 비율 계산 중...")
    avg_bone_ratios = defaultdict(dict)
    for slot_id in df['slot_id'].unique():
        person_df = df_pivot.loc[pd.IndexSlice[:, slot_id], :]
        ratios = defaultdict(list)
        for frame_idx in person_df.index.get_level_values('frame'):
            try:
                shoulder_l = person_df.loc[(frame_idx, slot_id), [('x', 11), ('y', 11), ('z', 11)]].values
                shoulder_r = person_df.loc[(frame_idx, slot_id), [('x', 12), ('y', 12), ('z', 12)]].values
                hip_l = person_df.loc[(frame_idx, slot_id), [('x', 23), ('y', 23), ('z', 23)]].values
                hip_r = person_df.loc[(frame_idx, slot_id), [('x', 24), ('y', 24), ('z', 24)]].values
                vis_shoulders = person_df.loc[(frame_idx, slot_id), [('visibility', 11), ('visibility', 12)]].values
                vis_hips = person_df.loc[(frame_idx, slot_id), [('visibility', 23), ('visibility', 24)]].values
                if np.all(vis_shoulders > 0.7) and np.all(vis_hips > 0.7):
                    torso_len = get_3d_distance((shoulder_l + shoulder_r) / 2, (hip_l + hip_r) / 2)
                    if torso_len < 10: continue
                    for bone_name, (start_idx, end_idx) in BONE_MAPPING.items():
                        p1_vis = person_df.loc[(frame_idx, slot_id), ('visibility', start_idx)]
                        p2_vis = person_df.loc[(frame_idx, slot_id), ('visibility', end_idx)]
                        if p1_vis > 0.7 and p2_vis > 0.7:
                            p1 = person_df.loc[(frame_idx, slot_id), [('x',start_idx),('y',start_idx),('z',start_idx)]].values
                            p2 = person_df.loc[(frame_idx, slot_id), [('x',end_idx),('y',end_idx),('z',end_idx)]].values
                            bone_len = get_3d_distance(p1, p2)
                            ratios[bone_name].append(bone_len / torso_len)
            except (KeyError, IndexError): continue
        for bone_name, ratio_list in ratios.items():
            if ratio_list: avg_bone_ratios[slot_id][bone_name] = np.mean(ratio_list)

    # 2. 제약 조건 적용 및 데이터 보정
    print("  - 2/3: 뼈 길이 및 대칭성 제약을 적용하여 스켈레톤 보정 중...")
    corrected_df_pivot = df_pivot.copy()
    for (frame_idx, slot_id), row in corrected_df_pivot.iterrows():
        # --- 상대 비율 제약 (뼈 길이 보정) ---
        try:
            shoulder_l = row[[('x', 11), ('y', 11), ('z', 11)]].values; shoulder_r = row[[('x', 12), ('y', 12), ('z', 12)]].values
            hip_l = row[[('x', 23), ('y', 23), ('z', 23)]].values; hip_r = row[[('x', 24), ('y', 24), ('z', 24)]].values
            current_torso_len = get_3d_distance((shoulder_l + shoulder_r) / 2, (hip_l + hip_r) / 2)
            if current_torso_len < 10: continue
        except (KeyError, IndexError): continue
        for bone_name, (start_idx, end_idx) in BONE_MAPPING.items():
            if bone_name not in avg_bone_ratios.get(slot_id, {}): continue
            target_ratio = avg_bone_ratios[slot_id][bone_name]
            target_length = current_torso_len * target_ratio
            p_start = row[[('x',start_idx),('y',start_idx),('z',start_idx)]].values
            p_end = row[[('x',end_idx),('y',end_idx),('z',end_idx)]].values
            current_length = get_3d_distance(p_start, p_end)
            if abs(current_length - target_length) > target_length * BONE_LENGTH_THRESHOLD_RATIO:
                direction_vec = (p_end - p_start) / (current_length + 1e-6)
                corrected_p_end = p_start + direction_vec * target_length
                row[('x',end_idx)], row[('y',end_idx)], row[('z',end_idx)] = corrected_p_end

        # 대칭성 제약 (가려진 부분 복원)
        shoulder_mid = (row[[('x', 11), ('y', 11), ('z', 11)]].values + row[[('x', 12), ('y', 12), ('z', 12)]].values) / 2
        for left_idx, right_idx in SYMMETRIC_PAIRS:
            left_vis = row.get(('visibility', left_idx), 0); right_vis = row.get(('visibility', right_idx), 0)
            
            if left_vis > VISIBILITY_THRESHOLD and right_vis < VISIBILITY_THRESHOLD:
                p_left = row[[('x',left_idx),('y',left_idx),('z',left_idx)]].values
                # 몸 중심축(어깨 중앙)을 기준으로 대칭점 계산
                p_right_estimated = shoulder_mid + (shoulder_mid - p_left)
                row[('x',right_idx)], row[('y',right_idx)], row[('z',right_idx)] = p_right_estimated
                row[('visibility', right_idx)] = 0.5 # 복원된 값이란 의미로 가시성 조정
            
            elif right_vis > VISIBILITY_THRESHOLD and left_vis < VISIBILITY_THRESHOLD:
                p_right = row[[('x',right_idx),('y',right_idx),('z',right_idx)]].values
                p_left_estimated = shoulder_mid + (shoulder_mid - p_right)
                row[('x',left_idx)], row[('y',left_idx)], row[('z',left_idx)] = p_left_estimated
                row[('visibility', left_idx)] = 0.5

    # 3. 보정된 데이터를 다시 Long Format으로 변환 후 저장
    print("  - 3/3: 보정된 데이터를 CSV 파일로 저장 중...")
    corrected_df_long = corrected_df_pivot.stack(level=1).reset_index()
    bbox_df = df[['frame', 'slot_id', 'bbox_x1', 'bbox_y1', 'bbox_x2', 'bbox_y2']].drop_duplicates()
    final_df = pd.merge(corrected_df_long, bbox_df, on=['frame', 'slot_id'])
    
    final_df.to_csv(output_path, index=False)
    print(f"\n후처리 2단계 완료! 모든 물리적/상황적 오류가 보정된 데이터가 '{output_path}'에 저장되었습니다.")

# --- 스크립트 실행 ---
if __name__ == "__main__":
    apply_human_model_constraints(INPUT_CSV_PATH, OUTPUT_CSV_PATH)


후처리 2단계 시작: 인체 모델 제약 적용 (최종 완전판)
  - 1/3: 각 인물의 평균 신체 비율 계산 중...
  - 2/3: 뼈 길이 및 대칭성 제약을 적용하여 스켈레톤 보정 중...
  - 3/3: 보정된 데이터를 CSV 파일로 저장 중...


C:\Users\human\AppData\Local\Temp\ipykernel_9476\582108643.py:121: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  corrected_df_long = corrected_df_pivot.stack(level=1).reset_index()



후처리 2단계 완료! 모든 물리적/상황적 오류가 보정된 데이터가 'TWICE_ICANTSTOPME_skeletons_data_final_corrected.csv'에 저장되었습니다.


In [22]:
# 후처리3
# Savitzky-Golay 필터
# 데이터의 작은 구간에 **다항식 곡선을 피팅(fitting)**하여 부드럽게 만드는 더 지능적인 방식

import pandas as pd
from scipy.signal import savgol_filter
import time

# --- 1. 설정 및 파라미터 ---
# 이전 단계('인체 모델 제약 적용')에서 생성된 데이터 파일
INPUT_CSV_PATH = 'TWICE_ICANTSTOPME_skeletons_data_final_corrected.csv'
# 최종적으로 스무딩된 데이터가 저장될 파일
OUTPUT_CSV_PATH = 'TWICE_ICANTSTOPME_final_smoothed.csv'

# Savitzky-Golay 필터 파라미터 (이 값들을 조정하여 스무딩 강도 조절 가능)
# 데이터 포인트를 몇 개나 보고 부드럽게 만들지 결정 (홀수, polyorder보다 커야 함)
WINDOW_LENGTH = 17
# 몇 차 다항식으로 피팅할지 결정 (값이 클수록 원본 데이터의 특징을 더 보존)
POLYORDER = 3     

# --- 2. 메인 후처리 로직 ---
def apply_global_smoothing(csv_path, output_path):
    """
    최종 보정된 데이터에 Savitzky-Golay 필터를 적용하여 미세한 떨림을 제거합니다.
    """
    start_time = time.time()
    print("후처리 3단계 시작: 전역적 스무딩 (최종 떨림 제거)")
    
    # 1. 데이터 로드
    df = pd.read_csv(csv_path)
    print(f"  - 원본 데이터 로드 완료: {len(df)} 행")

    # 스무딩할 좌표 컬럼
    coords_to_smooth = ['x', 'y', 'z']
    
    # 2. 각 ID, 각 관절의 이동 경로에 개별적으로 필터 적용
    # groupby와 transform을 사용하여 효율적으로 처리
    print(f"  - Savitzky-Golay 필터 적용 중 (window={WINDOW_LENGTH}, polyorder={POLYORDER})...")
    
    # 각 그룹의 데이터가 필터 윈도우보다 작을 경우를 대비한 예외 처리
    def smooth(series):
        if len(series) < WINDOW_LENGTH:
            # 윈도우보다 짧은 데이터는 스무딩하지 않고 원본 유지
            return series
        return savgol_filter(series, window_length=WINDOW_LENGTH, polyorder=POLYORDER)

    # slot_id와 landmark_id로 그룹화하여 각 시계열에 스무딩 적용
    df[coords_to_smooth] = df.groupby(['slot_id', 'landmark_id'])[coords_to_smooth].transform(smooth)
    
    # 3. 최종 데이터 저장
    df.to_csv(output_path, index=False)
    
    end_time = time.time()
    print(f"\n후처리 3단계 완료! (총 소요 시간: {end_time - start_time:.2f}초)")
    print(f"미세 떨림이 제거된 최종 데이터가 '{output_path}'에 저장되었습니다.")


# --- 3. 스크립트 실행 ---
if __name__ == "__main__":
    apply_global_smoothing(INPUT_CSV_PATH, OUTPUT_CSV_PATH)

후처리 3단계 시작: 전역적 스무딩 (최종 떨림 제거)
  - 원본 데이터 로드 완료: 32967 행
  - Savitzky-Golay 필터 적용 중 (window=17, polyorder=3)...

후처리 3단계 완료! (총 소요 시간: 0.49초)
미세 떨림이 제거된 최종 데이터가 'TWICE_ICANTSTOPME_final_smoothed.csv'에 저장되었습니다.


In [23]:
# 결과 확인용 시각화 스크립트
import cv2
import pandas as pd
import numpy as np
import mediapipe as mp
import os
import random

# --- 1. 설정 ---
# 최종 보정된 데이터 파일 경로
# ('인체 모델 제약 적용' 단계에서 생성된 최종 결과물)
FINAL_CSV_PATH = 'TWICE_ICANTSTOPME_final_smoothed.csv'

# 원본 비디오 파일 경로
VIDEO_PATH = 'TWICE_ICANTSTOPME.mp4'

# 최종 결과 영상이 저장될 경로
OUTPUT_VIDEO_PATH = 'TWICE_ICANTSTOPME_visualization_ver3.mp4'

# MediaPipe 관절 연결 정보 (뼈대를 그리기 위함)
POSE_CONNECTIONS = mp.solutions.pose.POSE_CONNECTIONS

# --- 2. 헬퍼 함수 ---
def get_color_for_id(slot_id):
    """ID별로 일관된 색상을 생성하기 위한 간단한 함수"""
    # ID 번호를 기반으로 색상 생성
    random.seed(slot_id)
    return (random.randint(0, 255), random.randint(0, 255), random.randint(0, 255))

# --- 3. 메인 시각화 로직 ---
def visualize_postprocessed_skeletons(csv_path, video_path, output_path):
    """
    최종 보정된 스켈레톤 데이터를 읽어 원본 영상 위에 그려,
    결과 동영상을 생성합니다.
    """
    # 1. 파일 존재 여부 확인
    if not os.path.exists(csv_path):
        print(f"오류: 데이터 파일 '{csv_path}'를 찾을 수 없습니다.")
        return
    if not os.path.exists(video_path):
        print(f"오류: 비디오 파일 '{video_path}'를 찾을 수 없습니다.")
        return

    print("시각화 시작: 최종 보정 데이터를 영상에 그립니다.")
    
    # 2. 데이터 로드 및 전처리
    # 프레임별로 데이터를 빠르게 조회하기 위해 'frame'을 인덱스로 설정
    df = pd.read_csv(csv_path)
    df.set_index('frame', inplace=True)
    
    # 3. 비디오 파일 및 결과 영상 설정
    cap = cv2.VideoCapture(video_path)
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (frame_width, frame_height))

    frame_counter = 0
    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break

        # 4. 현재 프레임에 해당하는 스켈레톤 데이터 조회
        try:
            frame_data = df.loc[frame_counter]
        except KeyError:
            # 해당 프레임에 데이터가 없으면 원본 프레임만 저장
            out.write(frame)
            frame_counter += 1
            continue

        # 5. 각 ID별로 스켈레톤 그리기
        # slot_id별로 데이터를 그룹화
        for slot_id, group in frame_data.groupby('slot_id'):
            landmarks = group.sort_values(by='landmark_id')
            
            # 랜드마크 좌표와 가시성 추출
            points = landmarks[['x', 'y']].values
            visibility = landmarks['visibility'].values
            
            color = get_color_for_id(int(slot_id))
            
            # 바운딩 박스 그리기
            bbox = group.iloc[0][['bbox_x1', 'bbox_y1', 'bbox_x2', 'bbox_y2']].values
            x1, y1, x2, y2 = [int(c) for c in bbox]
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, f"ID: {int(slot_id)}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

            # 관절(점) 그리기
            for i, point in enumerate(points):
                if visibility[i] > 0.5:
                    cv2.circle(frame, (int(point[0]), int(point[1])), 5, color, -1)
            
            # 뼈대(선) 그리기
            for connection in POSE_CONNECTIONS:
                start_idx, end_idx = connection
                if visibility[start_idx] > 0.5 and visibility[end_idx] > 0.5:
                    start_point = (int(points[start_idx, 0]), int(points[start_idx, 1]))
                    end_point = (int(points[end_idx, 0]), int(points[end_idx, 1]))
                    cv2.line(frame, start_point, end_point, color, 2)

        out.write(frame)
        frame_counter += 1
        
        # 실시간으로 보기 (선택사항)
        # cv2.imshow('Final Visualization', frame)
        # if cv2.waitKey(1) & 0xFF == ord('q'):
        #     break

    # 6. 종료 처리
    cap.release()
    out.release()
    cv2.destroyAllWindows()
    print(f"\n시각화 완료! 최종 결과 영상이 '{output_path}'에 저장되었습니다.")


# --- 4. 스크립트 실행 ---
if __name__ == "__main__":
    visualize_postprocessed_skeletons(FINAL_CSV_PATH, VIDEO_PATH, OUTPUT_VIDEO_PATH)

시각화 시작: 최종 보정 데이터를 영상에 그립니다.

시각화 완료! 최종 결과 영상이 'TWICE_ICANTSTOPME_visualization_ver3.mp4'에 저장되었습니다.


In [ ]:
!pip install pandas openpyxl

In [ ]:
# 롱 포멧 방식의 결과를 사람의 가독성을 위해 와이드 포멧으로 변경하는 코드
import pandas as pd
import os
import mediapipe as mp

# --- 설정 ---
# 변환할 원본 데이터 파일 경로
input_csv_path = 'aespa_test_skeletons_data_final_corrected.csv' 

# 최종적으로 생성될 엑셀 보고서 파일 경로
output_excel_path = input_csv_path.replace('_skeletons_data_final_corrected.csv', '_skeletons_report.xlsx')

# MediaPipe의 PoseLandmark enum을 사용하여, ID를 실제 관절 이름으로 매핑합니다.
landmark_names = [name.name for name in mp.solutions.pose.PoseLandmark]

# --- 데이터 변환 로직 ---
def convert_long_to_wide_excel(csv_path, excel_path):
    """
    '롱 포맷' CSV 데이터를 읽어, slot_id별로 시트를 나눈 '와이드 포맷' 엑셀 파일로 변환합니다.
    이때, 열을 '관절 부위' 중심으로 정렬합니다. (예: x_NOSE, y_NOSE, z_NOSE, v_NOSE, ...)
    """
    if not os.path.exists(csv_path):
        print(f"오류: 원본 데이터 파일 '{csv_path}'를 찾을 수 없습니다.")
        return

    print(f"'{csv_path}' 파일을 읽는 중입니다...")
    df_long = pd.read_csv(csv_path)
    
    track_ids = df_long['slot_id'].unique()
    
    print(f"총 {len(track_ids)}개의 고유한 Track ID를 발견했습니다: {sorted(track_ids)}")
    print(f"'{excel_path}' 엑셀 파일 생성을 시작합니다...")

    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
        for track_id in sorted(track_ids):
            print(f"  - Track ID: {track_id} 시트를 처리하는 중...")
            
            df_person = df_long[df_long['slot_id'] == track_id]
            
            df_wide = df_person.pivot_table(
                index='frame', 
                columns='landmark_id', 
                values=['x', 'y', 'z', 'visibility']
            )
            
            # 💡 [개선] 열 순서를 관절 중심으로 재정렬합니다.
            # 1. 원하는 순서대로 컬럼 리스트를 새로 생성합니다.
            ordered_columns_tuples = []
            for i in range(len(landmark_names)): # 0부터 32까지 관절 ID 순회
                for axis in ['x', 'y', 'z', 'visibility']: # 각 관절에 대해 x, y, z, v 순서로 추가
                    if (axis, i) in df_wide.columns: # 데이터에 실제 존재하는 컬럼인지 확인
                        ordered_columns_tuples.append((axis, i))

            # 2. 생성된 순서대로 DataFrame의 열을 재정렬합니다.
            df_wide = df_wide[ordered_columns_tuples]

            # 3. 재정렬된 컬럼의 이름을 직관적으로 변경합니다.
            new_columns = []
            for axis, idx in df_wide.columns:
                landmark_name = landmark_names[idx]
                axis_name = 'v' if axis == 'visibility' else axis
                new_columns.append(f'{axis_name}_{landmark_name}')
            
            df_wide.columns = new_columns
            
            df_wide = df_wide.sort_index()

            sheet_name = f'ID_{track_id}'
            df_wide.to_excel(writer, sheet_name=sheet_name)

    print("\n변환 완료!")
    print(f"최종 보고서가 '{excel_path}' 경로에 성공적으로 저장되었습니다.")


# --- 스크립트 실행 ---
if __name__ == "__main__":
    convert_long_to_wide_excel(input_csv_path, output_excel_path)

'aespa_test_skeletons_data.csv' 파일을 읽는 중입니다...
총 4개의 고유한 Track ID를 발견했습니다: [0, 1, 2, 3]
'aespa_test_skeletons_report.xlsx' 엑셀 파일 생성을 시작합니다...
  - Track ID: 0 시트를 처리하는 중...
  - Track ID: 1 시트를 처리하는 중...
  - Track ID: 2 시트를 처리하는 중...
  - Track ID: 3 시트를 처리하는 중...

변환 완료!
최종 보고서가 'aespa_test_skeletons_report.xlsx' 경로에 성공적으로 저장되었습니다.
